# US Oil Production Sites Explorer

End-to-end notebook for the `us_oil_production_sites_explorer` agent. Set the variables in the **Configuration** cell below, then run cells top-to-bottom.

Phases:

1. **Schema migration** — idempotent. Creates `production_sites.*` if missing.
2. **Bootstrap** — single agent invocation enumerating ~300–500 top US oil fields from EIA reports, state regulators (TX RRC, ND DMR, AK AOGCC, CO COGCC, NM OCD, BSEE), and operator 10-Ks. Writes to `production_sites.field` and `outputs/production_sites/seed_list.json`.
3. **Field-by-field exploration** — loop over the chosen target set; each field gets all 9 child-table rows plus `outputs/production_sites/<field_id>/profile.json`.

Resume-aware: cells skip work that's already done (seed populated, field already explored) unless their FORCE flag is set.

**Prerequisites**: `pip install claude-agent-sdk`; `claude login` (subscription auth — no `ANTHROPIC_API_KEY` needed).

## Configuration

Everything you'd want to change between runs lives in this one cell.

In [ ]:
# %% Configuration — edit here, then run all cells.

# ---- Models ---------------------------------------------------------------- #
BOOTSTRAP_MODEL = 'claude-sonnet-4-6'    # only used if SEED_SOURCE = 'agent'
EXPLORE_MODEL   = 'claude-sonnet-4-6'    # per-field deep research

# ---- Seed source ----------------------------------------------------------- #
#   'json'  — deterministic, instant, idempotent (Stage2/config/production_sites_seed.json)
#   'agent' — LLM-driven enumeration via bootstrap_seed_async (~$1–5, ~10 min)
SEED_SOURCE = 'json'

# ---- Mode: how many fields to explore -------------------------------------- #
#   'single' | 'calibration' | 'basin_sample' | 'full'
MODE = 'single'

# ---- Resume / re-run behaviour --------------------------------------------- #
FORCE_BOOTSTRAP = False    # re-run Phase 1 even if seed is populated
FORCE_EXPLORE   = False    # re-run Phase 2 even on already-explored fields

# ---- Persistence mode ------------------------------------------------------ #
#   False — APPEND new run's rows to the child tables, tagged with the new
#           run_id; prior runs preserved. Post-persist consistency scan
#           detects "same logical entity across runs" and writes findings to
#           exploration_runs.inconsistencies as JSONB.
#   True  — REPLACE: delete every existing child row for this field_id
#           BEFORE the new run inserts. exploration_runs audit log preserved.
REPLACE_EXISTING = False

# ---- Calibration set (used when MODE in ('single', 'calibration')) --------- #
CALIBRATION_SLUGS = [
    'spraberry_trend_tx',   # Permian tight oil
    'prudhoe_bay_ak',       # Alaska North Slope arctic conventional
    'mars_ursa_gom',        # GoM deepwater (federal offshore)
]

print(f'seed_source       = {SEED_SOURCE!r}')
print(f'mode              = {MODE!r}')
print(f'bootstrap_model   = {BOOTSTRAP_MODEL!r}')
print(f'explore_model     = {EXPLORE_MODEL!r}')
print(f'force_bootstrap   = {FORCE_BOOTSTRAP}')
print(f'force_explore     = {FORCE_EXPLORE}')
print(f'replace_existing  = {REPLACE_EXISTING}')
print(f'calibration_slugs = {CALIBRATION_SLUGS}')


## Setup — imports + path

In [2]:
# %% Imports
import os, sys, json, time
from collections import Counter
from pathlib import Path

import psycopg2

# code/ is the working dir; make sure it's importable.
CODE_DIR = Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from us_oil_production_sites_explorer import (
    bootstrap_seed_async,
    load_seed_from_json,
    list_fields,
    explore_field_async,
    persist_to_db,
    already_explored,
    notify_done,
    PRODUCTION_SITES_OUT,
)

# Pull the DDL from the migration so we can apply it in-notebook.
from migrations.create_production_sites_schema import DDL as PRODUCTION_SITES_DDL

_DB_KW = dict(host='localhost', dbname='eia_crude',
              user='eia_user', password='eia_password')

PRODUCTION_SITES_OUT.mkdir(parents=True, exist_ok=True)
print('output dir:', PRODUCTION_SITES_OUT)

output dir: C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Stage2\outputs\production_sites


## Phase 0 — Schema migration

Idempotent. Creates the 9 `production_sites.*` tables if they don't exist.

In [3]:
# %% Apply the schema DDL (idempotent)
with psycopg2.connect(**_DB_KW) as _conn:
    with _conn.cursor() as cur:
        cur.execute(PRODUCTION_SITES_DDL)
    _conn.commit()
    with _conn.cursor() as cur:
        cur.execute(
            "SELECT table_name FROM information_schema.tables "
            "WHERE table_schema = 'production_sites' ORDER BY table_name"
        )
        tables = [r[0] for r in cur.fetchall()]

print('production_sites tables present:')
for t in tables:
    print(f'  - production_sites.{t}')

production_sites tables present:
  - production_sites.events
  - production_sites.exploration_runs
  - production_sites.field
  - production_sites.field_grades
  - production_sites.field_operators
  - production_sites.logistics_links
  - production_sites.production_history
  - production_sites.reserves
  - production_sites.sources


## Phase 1 — Bootstrap the seed list

Skips automatically if the seed is already populated. Set `FORCE_BOOTSTRAP = True` in the Configuration cell to re-run (appends new fields; existing rows are kept via `ON CONFLICT DO NOTHING`).

In [4]:
# %% Bootstrap seed list (Phase 1)
existing = list_fields()
if existing and not FORCE_BOOTSTRAP:
    print(f'seed already populated: {len(existing)} fields. Skipping bootstrap.')
    print(f'  set FORCE_BOOTSTRAP=True in the Configuration cell to re-run.')
elif SEED_SOURCE == 'json':
    print('loading seed deterministically from Stage2/config/production_sites_seed.json ...')
    res = load_seed_from_json()
    print(f'  entries in JSON: {res["entries"]}')
    print(f'  inserted:        {res["inserted"]}')
    print(f'  total in DB:     {res["after"]}')
elif SEED_SOURCE == 'agent':
    print(f'running agent bootstrap with model={BOOTSTRAP_MODEL!r} ...')
    t0 = time.time()
    seed_buf, seed_meta = await bootstrap_seed_async(model=BOOTSTRAP_MODEL)
    elapsed = time.time() - t0
    print(f'  status={seed_meta["status"]}  tool_calls={seed_meta["tool_calls"]}  '
          f'cost=${seed_meta["cost_usd"]:.3f}  elapsed={elapsed:.0f}s')
    print(f'  fields enumerated: {len(seed_buf["fields"])}')
    if seed_buf['summary']:
        print(f'  summary: {seed_buf["summary"]}')
    print()
    print(f'seed_list.json written to: {PRODUCTION_SITES_OUT / "seed_list.json"}')
    print(f'rows in production_sites.field: {len(list_fields())}')
else:
    raise ValueError(f'unknown SEED_SOURCE={SEED_SOURCE!r} — use "json" or "agent"')

seed already populated: 136 fields. Skipping bootstrap.
  set FORCE_BOOTSTRAP=True in the Configuration cell to re-run.


In [5]:
# %% Inspect the seed list grouped by basin
fields = list_fields()
print(f'total fields in seed: {len(fields)}')
print()
by_basin = Counter(f['basin'] or '(unknown)' for f in fields)
print('by basin:')
for basin, n in sorted(by_basin.items(), key=lambda x: -x[1]):
    print(f'  {n:4}  {basin}')
print()
print('first 10 fields:')
for f in fields[:10]:
    print(f'  {f["field_id"]:45} {(f["state"] or "--"):4} {(f["basin"] or "-"):20} {f["name"]}')

total fields in seed: 136

by basin:
    29  Permian
    23  Gulf of Mexico
    20  Williston
    10  Alaska North Slope
    10  Eagle Ford
    10  San Joaquin
     7  DJ/Niobrara
     6  Anadarko/SCOOP/STACK
     6  Powder River
     5  Alaska Cook Inlet
     4  Los Angeles
     3  Uinta
     2  Piceance
     1  Salinas

first 10 fields:
  granite_point_ak                              AK   Alaska Cook Inlet    Granite Point
  mcarthur_river_ak                             AK   Alaska Cook Inlet    McArthur River
  middle_ground_shoal_ak                        AK   Alaska Cook Inlet    Middle Ground Shoal
  swanson_river_ak                              AK   Alaska Cook Inlet    Swanson River
  trading_bay_ak                                AK   Alaska Cook Inlet    Trading Bay
  alpine_ak                                     AK   Alaska North Slope   Alpine
  endicott_ak                                   AK   Alaska North Slope   Endicott
  kuparuk_river_ak                              AK

## Phase 2 — Field-by-field deep research

Target set is derived from `MODE` set in the Configuration cell. Resume-aware: already-explored fields are skipped unless `FORCE_EXPLORE=True`.

In [6]:
# %% Pick target fields based on MODE
fields = list_fields()

if MODE == 'single':
    targets = [f for f in fields if f['field_id'] == CALIBRATION_SLUGS[0]]
    if not targets:
        raise ValueError(f'CALIBRATION_SLUGS[0]={CALIBRATION_SLUGS[0]!r} not in seed.')
elif MODE == 'calibration':
    targets = [f for f in fields if f['field_id'] in CALIBRATION_SLUGS]
    missing = set(CALIBRATION_SLUGS) - {f['field_id'] for f in targets}
    if missing:
        print(f'WARNING: calibration slugs not in seed: {missing}')
        if not targets:
            print(f'  fallback: take the first 3 fields from the seed.')
            targets = fields[:3]
elif MODE == 'basin_sample':
    by_basin_lst: dict[str, list] = {}
    for f in fields:
        b = f['basin'] or '(unknown)'
        by_basin_lst.setdefault(b, []).append(f)
    targets = []
    for b, lst in by_basin_lst.items():
        targets.extend(lst[:2])
elif MODE == 'full':
    targets = fields
else:
    raise ValueError(f'unknown MODE={MODE!r} — use single|calibration|basin_sample|full')

print(f'mode={MODE!r}  model={EXPLORE_MODEL!r}  force={FORCE_EXPLORE}  targets={len(targets)}')
print()
for f in targets[:20]:
    print(f'  {f["field_id"]:45} {(f["state"] or "--"):4} {(f["basin"] or "-"):20} {f["name"]}')
if len(targets) > 20:
    print(f'  ... ({len(targets) - 20} more)')

mode='single'  model='claude-sonnet-4-6'  force=False  targets=1

  spraberry_trend_tx                            TX   Permian              Spraberry Trend Area


In [ ]:
# %% Run the agent across the chosen targets — via subprocess.
# Why subprocess: on Windows, Jupyter forces SelectorEventLoop for zmq, but
# claude-agent-sdk's asyncio.create_subprocess_exec needs ProactorEventLoop
# to spawn its bundled claude.exe. So we shell out to run_one_field.py
# (which runs under the default ProactorEventLoop and works correctly).
import subprocess

LAUNCHER = CODE_DIR / 'run_one_field.py'
PYTHON   = Path(sys.executable)

summary = []
for i, f in enumerate(targets, 1):
    fid = f['field_id']
    if not FORCE_EXPLORE and already_explored(fid):
        print(f'[{i}/{len(targets)}] skipped (already explored): {fid}')
        continue

    cmd = [str(PYTHON), str(LAUNCHER), fid, '--model', EXPLORE_MODEL]
    if FORCE_EXPLORE:    cmd.append('--force')
    if REPLACE_EXISTING: cmd.append('--replace')

    print(f'[{i}/{len(targets)}] {fid}: {f["name"]}')
    print(f'    cmd: {" ".join(cmd)}')
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    elapsed = time.time() - t0
    if proc.stdout:
        for line in proc.stdout.splitlines():
            print(f'    {line}')
    if proc.returncode != 0:
        print(f'    EXIT {proc.returncode}')
        if proc.stderr:
            print(f'    stderr (truncated): {proc.stderr[:500]}')
        summary.append({'field_id': fid, 'name': f['name'], 'status': 'failed',
                        'exit_code': proc.returncode, 'elapsed': elapsed})
    else:
        summary.append({'field_id': fid, 'name': f['name'], 'status': 'ok',
                        'elapsed': elapsed})

print()
print('done. per-field summary:')
for s in summary:
    print(f'  {s.get("field_id"):45} {s.get("status"):8} elapsed={s.get("elapsed", 0):.0f}s')

notify_done(summary, mode=MODE)


## Inspect — sanity-check one field's outputs

In [8]:
# %% Inspect rows + profile.json for the first target
INSPECT_ID = targets[0]['field_id'] if targets else None

if INSPECT_ID:
    with psycopg2.connect(**_DB_KW) as conn:
        for table in ('field', 'field_operators', 'field_grades', 'production_monthly',
                      'reserves', 'events', 'logistics_links', 'sources', 'exploration_runs'):
            with conn.cursor() as cur:
                cur.execute(f'SELECT COUNT(*) FROM production_sites.{table} WHERE field_id = %s', (INSPECT_ID,))
                n = cur.fetchone()[0]
            print(f'  production_sites.{table:22} rows for {INSPECT_ID}: {n}')

    profile_path = PRODUCTION_SITES_OUT / INSPECT_ID / 'profile.json'
    if profile_path.exists():
        print(f'\nprofile.json at {profile_path}:')
        profile = json.loads(profile_path.read_text())
        print(f'  summary: {profile["result"]["summary"]}')
        print(f'  sources         : {len(profile["result"]["sources"])}')
        print(f'  operators       : {len(profile["result"]["operators"])}')
        print(f'  grades          : {len(profile["result"]["grades"])}')
        print(f'  production rows : {len(profile["result"]["production"])}')
        print(f'  reserves        : {len(profile["result"]["reserves"])}')
        print(f'  events          : {len(profile["result"]["events"])}')
        print(f'  logistics       : {len(profile["result"]["logistics"])}')
else:
    print('no targets to inspect.')

  production_sites.field                  rows for spraberry_trend_tx: 1
  production_sites.field_operators        rows for spraberry_trend_tx: 0
  production_sites.field_grades           rows for spraberry_trend_tx: 0
  production_sites.production_history     rows for spraberry_trend_tx: 0
  production_sites.reserves               rows for spraberry_trend_tx: 0
  production_sites.events                 rows for spraberry_trend_tx: 0
  production_sites.logistics_links        rows for spraberry_trend_tx: 0
  production_sites.sources                rows for spraberry_trend_tx: 0
  production_sites.exploration_runs       rows for spraberry_trend_tx: 1

profile.json at C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Stage2\outputs\production_sites\spraberry_trend_tx\profile.json:
  summary: None
  sources         : 0
  operators       : 0
  grades          : 0
  production rows : 0
  reserves        : 0
  events          : 0
  logistics       : 0


## Inspect — canonical views (latest-run row per natural key)

Use these views when you want the "current best answer" without cross-run noise — they pick the row from the highest `run_id` per natural key (same operator+start_year, same event_type+start_date, etc.).

In [ ]:
# %% Canonical views — latest-run row per natural key
import pandas as pd

INSPECT_ID = (targets[0]['field_id'] if targets else None)
if INSPECT_ID:
    with psycopg2.connect(**_DB_KW) as conn:
        for view in ('v_canonical_field_operators', 'v_canonical_field_grades', 'v_canonical_events'):
            df = pd.read_sql_query(
                f"SELECT * FROM production_sites.{view} WHERE field_id = %(fid)s ORDER BY 1",
                conn, params={'fid': INSPECT_ID})
            print(f'--- production_sites.{view} ({len(df)} rows) ---')
            print(df.head(10).to_string(index=False))
            print()
else:
    print('no targets to inspect.')


## Inspect — run inconsistencies

The post-persist consistency scan flags "same logical entity emitted under multiple runs". Findings land in `exploration_runs.inconsistencies` (JSONB) and are flattened by `v_run_inconsistencies`.

In [ ]:
# %% Inconsistencies across runs for the inspected field
import pandas as pd

if INSPECT_ID:
    with psycopg2.connect(**_DB_KW) as conn:
        df = pd.read_sql_query(
            """
            SELECT run_id, table_name, natural_key, n_conflicting_rows, run_ids_involved
            FROM production_sites.v_run_inconsistencies
            WHERE field_id = %(fid)s
            ORDER BY run_id DESC, table_name, natural_key
            """,
            conn, params={'fid': INSPECT_ID})
    if df.empty:
        print(f'no inconsistencies recorded for {INSPECT_ID}')
    else:
        print(f'inconsistencies recorded for {INSPECT_ID}:')
        print(df.to_string(index=False))


## Inspect — agent self-flagged caveats

Issues the agent reported via the `report_issue` tool — data gaps, source 403s, low-confidence figures, conflicting sources.

In [ ]:
# %% Agent self-flagged caveats via report_issue
import pandas as pd

if INSPECT_ID:
    with psycopg2.connect(**_DB_KW) as conn:
        df = pd.read_sql_query(
            """
            SELECT run_id, started_at, issue_type, severity, description
            FROM production_sites.v_run_agent_notes
            WHERE field_id = %(fid)s
            ORDER BY run_id DESC
            """,
            conn, params={'fid': INSPECT_ID})
    if df.empty:
        print(f'no agent_notes recorded for {INSPECT_ID}')
    else:
        print(f'agent_notes for {INSPECT_ID}:')
        for _, r in df.iterrows():
            print(f'  [{r.severity}] {r.issue_type}: {r.description}')
